# Batch 3 Kaggle Prep — Simple Colab Pipeline

This notebook is a **simplified version** of the larger ETL flow.

It is designed for current goal:

1. download the four Kaggle datasets,
2. keep the relevant CSV files,
3. run a light cleaning/preprocessing pass,
4. export **Batch 3-ready** files,
5. zip them so you can download them from Colab

**Important:** do not hardcode your Kaggle token in the notebook. Use the runtime prompt cell below.


In [ ]:
# ============================================================
# CELL 1: Install Kaggle + prompt for token + create folders
# ============================================================

# Step 1:
# Install the Kaggle CLI.
!pip -q install kaggle

# Step 2:
# Prompt for your Kaggle API token at runtime.
# Do NOT paste the token directly into the notebook source.
import os
import getpass
from pathlib import Path

os.environ["KAGGLE_API_TOKEN"] = getpass.getpass("Enter your Kaggle API token: ")

# Step 3:
# Create a simple folder structure inside the Colab runtime.
# We are keeping everything inside /content so this stays simple.
BASE = Path("/content/batch3_kaggle")
RAW = BASE / "raw"
UNZIPPED = BASE / "unzipped"
READY = BASE / "batch3_ready"
ZIPS = BASE / "zips"

for folder in [RAW, UNZIPPED, READY, ZIPS]:
    folder.mkdir(parents=True, exist_ok=True)

print("Folders created:")
print("RAW     =", RAW)
print("UNZIPPED=", UNZIPPED)
print("READY   =", READY)
print("ZIPS    =", ZIPS)

# Step 4:
# Quick real auth check.
# This should print the files for one dataset if auth worked.
!kaggle datasets files paultimothymooney/stock-market-data


Enter your Kaggle API token: ··········
Folders created:
RAW     = /content/batch3_kaggle/raw
UNZIPPED= /content/batch3_kaggle/unzipped
READY   = /content/batch3_kaggle/batch3_ready
ZIPS    = /content/batch3_kaggle/zips
Next Page Token = CfDJ8CS0IeAoHcJGgSEc27rBbk4RkIhYFe6cS6fRB0Z_30hHL84VNGIXXtUIZDwp1DMZ-OFL7n65qfQKiUcw9JWpWBRjjy916Ut1dFDDQI1_O_NtZWeZGY4DwNdRiwqQorqQgEUvljeFycw4BZF7SF4Wm1GUUpLQcSgNg-1OfRiVvKk
name                                           size  creationDate                
------------------------------------------  -------  --------------------------  
stock_market_data/forbes2000/csv/A.csv       638996  2023-01-16 21:04:05.439000  
stock_market_data/forbes2000/csv/AAALY.csv   257045  2023-01-16 21:04:05.452000  
stock_market_data/forbes2000/csv/AACAY.csv   312258  2023-01-16 21:04:05.467000  
stock_market_data/forbes2000/csv/AAL.csv     466378  2023-01-16 21:04:05.483000  
stock_market_data/forbes2000/csv/AAP.csv     568177  2023-01-16 21:04:05.476000  
stock_market

In [ ]:
# ============================================================
# CELL 2: Download and unzip the four Kaggle datasets
# ============================================================

# Step 1:
# Define the four dataset slugs we want.
DATASETS = {
    "01_stock_market_data": "paultimothymooney/stock-market-data",
    "02_sp500_financials_news": "sadiqguru/s-and-p-500-stock-data-along-with-financials-and-news",
    "03_sp500_companies_financial_info": "paytonfisher/sp-500-companies-with-financial-information",
    "04_sp500_stocks": "andrewmvd/sp-500-stocks",
}

# Step 2:
# Download each dataset zip and unzip it into its own folder.
for short_name, slug in DATASETS.items():
    out_zip_dir = RAW / short_name
    out_unzip_dir = UNZIPPED / short_name
    out_zip_dir.mkdir(parents=True, exist_ok=True)
    out_unzip_dir.mkdir(parents=True, exist_ok=True)

    print(f"\nDownloading: {slug}")
    !kaggle datasets download -d {slug} -p {str(out_zip_dir)} --unzip

    # Note:
    # We are using --unzip directly, so the files should already be extracted
    # into out_zip_dir. To keep paths simple, we will treat RAW/<dataset> as the
    # extracted folder for inventory + cleaning.
print("\nAll four datasets downloaded.")



Downloading: paultimothymooney/stock-market-data
Dataset URL: https://www.kaggle.com/datasets/paultimothymooney/stock-market-data
License(s): other
100% 1.03G/1.03G [00:13<00:00, 80.6MB/s]


Downloading: sadiqguru/s-and-p-500-stock-data-along-with-financials-and-news
Dataset URL: https://www.kaggle.com/datasets/sadiqguru/s-and-p-500-stock-data-along-with-financials-and-news
License(s): MIT
100% 76.4M/76.4M [00:00<00:00, 112MB/s]


Downloading: paytonfisher/sp-500-companies-with-financial-information
Dataset URL: https://www.kaggle.com/datasets/paytonfisher/sp-500-companies-with-financial-information
License(s): ODC Public Domain Dedication and Licence (PDDL)
100% 29.5k/29.5k [00:00<00:00, 29.8MB/s]


Downloading: andrewmvd/sp-500-stocks
Dataset URL: https://www.kaggle.com/datasets/andrewmvd/sp-500-stocks
License(s): CC0-1.0
100% 18.7M/18.7M [00:00<00:00, 71.7MB/s]


All four datasets downloaded.


In [ ]:
# ============================================================
# CELL 3: Inventory the files so you can see what exists
# ============================================================

# Step 1:
# Walk through each dataset folder and record file names + sizes.
import os
import pandas as pd

def inventory(root_dir):
    rows = []
    for dirpath, _, filenames in os.walk(root_dir):
        for fname in filenames:
            full_path = os.path.join(dirpath, fname)
            rows.append({
                "relative_path": os.path.relpath(full_path, root_dir),
                "size_mb": round(os.path.getsize(full_path) / (1024 * 1024), 2),
            })
    df = pd.DataFrame(rows)
    if len(df) == 0:
        return df
    return df.sort_values(["relative_path"]).reset_index(drop=True)

inventories = {}

for short_name in DATASETS:
    folder = RAW / short_name
    inv = inventory(folder)
    inventories[short_name] = inv

    print("\n" + "=" * 80)
    print(short_name)
    print("=" * 80)
    display(inv)

    # Save the inventory CSV into the ready folder too.
    out_dir = READY / short_name
    out_dir.mkdir(parents=True, exist_ok=True)
    inv.to_csv(out_dir / f"{short_name}_inventory.csv", index=False)



01_stock_market_data


,relative_path,size_mb
0,stock_market_data/forbes2000/csv/A.csv,0.61
1,stock_market_data/forbes2000/csv/AAALY.csv,0.25
2,stock_market_data/forbes2000/csv/AACAY.csv,0.30
3,stock_market_data/forbes2000/csv/AAL.csv,0.44
4,stock_market_data/forbes2000/csv/AAP.csv,0.54
...,...,...
9205,stock_market_data/sp500/json/XYL.json,0.88
9206,stock_market_data/sp500/json/YUM.json,2.01
9207,stock_market_data/sp500/json/ZBH.json,1.70
9208,stock_market_data/sp500/json/ZION.json,3.15



02_sp500_financials_news


,relative_path,size_mb
0,01_company_info.csv,1.15
1,02_company_news_sentiment.csv,10.23
2,03_company_balance_sheet.csv,7.62
3,04_company_financials.csv,4.99
4,05_company_filings.csv,4.67
...,...,...
529,price_data/YUM.csv,0.38
530,price_data/ZBH.csv,0.38
531,price_data/ZBRA.csv,0.37
532,price_data/ZION.csv,0.38



03_sp500_companies_financial_info


,relative_path,size_mb
0,financials.csv,0.09



04_sp500_stocks


,relative_path,size_mb
0,sp500_companies.csv,0.77
1,sp500_index.csv,0.05
2,sp500_stocks.csv,91.85


In [ ]:
# ============================================================
# CELL 4: Decide which CSV files are relevant
# ============================================================

# Step 1:
# This helper finds CSV files recursively.
from pathlib import Path

def find_csvs(root_dir):
    return sorted([str(p) for p in Path(root_dir).rglob("*.csv")])

# Step 2:
# Build a simple default selection rule:
# - Dataset 1: keep only CSV files whose path contains 'sp500'
# - Datasets 2/3/4: keep all CSV files
#
# This keeps the notebook simple and aligned with your current goal.
selected_files = {}

for short_name in DATASETS:
    csvs = find_csvs(RAW / short_name)

    if short_name == "01_stock_market_data":
        selected = [p for p in csvs if "sp500" in p.lower()]
    else:
        selected = csvs

    selected_files[short_name] = selected

# Step 3:
# Print the selected files so you can quickly sanity-check them.
for short_name, files_list in selected_files.items():
    print("\n" + "-" * 80)
    print(f"Selected CSV files for {short_name}:")
    print("-" * 80)
    for p in files_list:
        print(p)

# Optional manual override:
# If you want, you can replace any selected_files[...] list manually after this cell runs.
# Example:
# selected_files["02_sp500_financials_news"] = [
#     "/content/batch3_kaggle/raw/02_sp500_financials_news/company_news_sentiment.csv"
# ]



--------------------------------------------------------------------------------
Selected CSV files for 01_stock_market_data:
--------------------------------------------------------------------------------
/content/batch3_kaggle/raw/01_stock_market_data/stock_market_data/sp500/csv/A.csv
/content/batch3_kaggle/raw/01_stock_market_data/stock_market_data/sp500/csv/AAL.csv
/content/batch3_kaggle/raw/01_stock_market_data/stock_market_data/sp500/csv/AAP.csv
/content/batch3_kaggle/raw/01_stock_market_data/stock_market_data/sp500/csv/AAPL.csv
/content/batch3_kaggle/raw/01_stock_market_data/stock_market_data/sp500/csv/ABBV.csv
/content/batch3_kaggle/raw/01_stock_market_data/stock_market_data/sp500/csv/ABC.csv
/content/batch3_kaggle/raw/01_stock_market_data/stock_market_data/sp500/csv/ABMD.csv
/content/batch3_kaggle/raw/01_stock_market_data/stock_market_data/sp500/csv/ABT.csv
/content/batch3_kaggle/raw/01_stock_market_data/stock_market_data/sp500/csv/ACN.csv
/content/batch3_kaggle/raw/01_stock

In [ ]:
# ============================================================
# CELL 5: Simple cleaning helpers
# ============================================================

# This cell mirrors the spirit of your larger notebook:
# read data -> clean it -> write cleaned outputs.
#
# We are intentionally skipping the parquet step here because your
# immediate goal is Batch 3 upload, not a giant warehouse rewrite.

import re
import numpy as np

def snake_case(name):
    # Convert column names to lowercase snake_case.
    name = str(name).strip()
    name = re.sub(r"[^A-Za-z0-9]+", "_", name)
    name = re.sub(r"_+", "_", name)
    return name.strip("_").lower()

def normalize_ticker(x):
    # Standardize ticker-like values:
    # - uppercase
    # - strip spaces
    # - unify some punctuation
    if pd.isna(x):
        return x
    x = str(x).strip().upper()
    x = x.replace(".", "-").replace("/", "-").replace(" ", "")
    return x

def maybe_parse_dates(df):
    # Parse columns that look date-like.
    for col in df.columns:
        if any(key in col for key in ["date", "published", "time", "timestamp"]):
            try:
                df[col] = pd.to_datetime(df[col], errors="coerce")
            except Exception:
                pass
    return df

def normalize_strings(df):
    # Strip whitespace from object columns.
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].astype(str).str.strip()
            df.loc[df[col].isin(["nan", "None", "NaT"]), col] = np.nan
    return df

def normalize_ticker_columns(df):
    # Normalize likely ticker/symbol columns.
    ticker_like = [c for c in df.columns if c in {"ticker", "symbol", "ticker_in_source"} or "ticker" in c or "symbol" in c]
    for col in ticker_like:
        df[col] = df[col].apply(normalize_ticker)
    return df

def clean_price_like(df):
    # Apply the notebook-style price cleaning:
    # - drop rows missing core fields if present
    # - remove negative numeric values from price/volume fields
    # - sort by ticker/date if present
    essential_candidates = [c for c in ["date", "trading_date", "close", "volume"] if c in df.columns]
    if essential_candidates:
        df = df.dropna(subset=essential_candidates)

    numeric_cols = [c for c in ["open", "high", "low", "close", "adj_close", "adjusted_close", "volume"] if c in df.columns]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df = df[df[col].isna() | (df[col] >= 0)]

    sort_cols = [c for c in ["ticker", "symbol", "date", "trading_date"] if c in df.columns]
    if sort_cols:
        df = df.sort_values(sort_cols)

    return df

def clean_company_like(df):
    # Deduplicate likely company reference rows.
    preferred_keys = []
    for key_set in [["ticker"], ["symbol"], ["ticker", "company_name"], ["symbol", "company_name"]]:
        if all(k in df.columns for k in key_set):
            preferred_keys = key_set
            break
    if preferred_keys:
        df = df.drop_duplicates(subset=preferred_keys)
    else:
        df = df.drop_duplicates()
    return df

def clean_news_like(df):
    # Basic news cleaning:
    # - sort by published time if present
    # - dedupe by url if present, otherwise by title+published_at if possible
    sort_cols = [c for c in ["published_at", "date", "published_date"] if c in df.columns]
    if sort_cols:
        df = df.sort_values(sort_cols)

    if "url" in df.columns:
        df = df.drop_duplicates(subset=["url"])
    elif all(c in df.columns for c in ["title", "published_at"]):
        df = df.drop_duplicates(subset=["title", "published_at"])
    else:
        df = df.drop_duplicates()

    return df

def classify_table(df):
    cols = set(df.columns)
    if {"open", "high", "low", "close"} & cols:
        return "price"
    if {"title", "url", "published_at"} & cols:
        return "news"
    if {"sector", "industry", "company_name", "ticker", "symbol"} & cols:
        return "company"
    return "generic"

def basic_clean(df):
    # 1) normalize column names
    df.columns = [snake_case(c) for c in df.columns]

    # 2) trim strings
    df = normalize_strings(df)

    # 3) parse date-like columns
    df = maybe_parse_dates(df)

    # 4) normalize ticker-like columns
    df = normalize_ticker_columns(df)

    # 5) remove exact duplicates first
    df = df.drop_duplicates()

    # 6) apply table-specific cleaning
    table_type = classify_table(df)
    if table_type == "price":
        df = clean_price_like(df)
    elif table_type == "company":
        df = clean_company_like(df)
    elif table_type == "news":
        df = clean_news_like(df)
    else:
        df = df.drop_duplicates()

    return df, table_type


In [ ]:
# ============================================================
# CELL 6: Clean the selected files and export Batch 3-ready outputs
# ============================================================

# Output strategy:
# For every selected CSV file, create:
# - a cleaned SAMPLE file (first 300 rows) for easy upload
# - a cleaned FULL file if it is small enough
# - cleaned CHUNKS if the full file is too large
# - a small metadata row in a per-dataset cleaning log

MAX_FULL_MB = 45
SAMPLE_ROWS = 300
CHUNK_ROWS = 150000

def file_size_mb(path):
    return os.path.getsize(path) / (1024 * 1024)

def save_chunked_csv(df, base_path, rows_per_chunk=150000):
    for i in range(0, len(df), rows_per_chunk):
        chunk = df.iloc[i:i + rows_per_chunk]
        chunk_path = f"{base_path}_part{i // rows_per_chunk + 1:02d}.csv"
        chunk.to_csv(chunk_path, index=False)

for short_name, files_list in selected_files.items():
    out_dir = READY / short_name
    out_dir.mkdir(parents=True, exist_ok=True)

    log_rows = []

    for file_path in files_list:
        try:
            src = Path(file_path)
            print(f"\nProcessing: {src}")

            # Step 1: read CSV
            df = pd.read_csv(src)

            # Step 2: clean it
            cleaned, table_type = basic_clean(df)

            # Step 3: build output names
            stem = snake_case(src.stem)
            sample_path = out_dir / f"{stem}__sample.csv"
            full_path = out_dir / f"{stem}__cleaned.csv"

            # Step 4: always save a sample
            cleaned.head(SAMPLE_ROWS).to_csv(sample_path, index=False)

            # Step 5: save full cleaned file if small enough; otherwise chunk it
            temp_full = out_dir / f"{stem}__temp_full.csv"
            cleaned.to_csv(temp_full, index=False)
            full_size = file_size_mb(temp_full)

            if full_size <= MAX_FULL_MB:
                temp_full.rename(full_path)
                saved_mode = "full"
            else:
                temp_full.unlink(missing_ok=True)
                save_chunked_csv(cleaned, str(out_dir / f"{stem}__chunk"), rows_per_chunk=CHUNK_ROWS)
                saved_mode = "chunked"

            log_rows.append({
                "source_file": str(src),
                "table_type_guess": table_type,
                "raw_rows": len(df),
                "cleaned_rows": len(cleaned),
                "sample_file": sample_path.name,
                "saved_mode": saved_mode,
            })

        except Exception as e:
            log_rows.append({
                "source_file": str(file_path),
                "table_type_guess": "ERROR",
                "raw_rows": None,
                "cleaned_rows": None,
                "sample_file": None,
                "saved_mode": f"error: {e}",
            })
            print(f"Error processing {file_path}: {e}")

    # Save per-dataset cleaning log
    pd.DataFrame(log_rows).to_csv(out_dir / f"{short_name}_cleaning_log.csv", index=False)

print("\nBatch 3-ready files created.")



Processing: /content/batch3_kaggle/raw/01_stock_market_data/stock_market_data/sp500/csv/A.csv

Processing: /content/batch3_kaggle/raw/01_stock_market_data/stock_market_data/sp500/csv/AAL.csv

/tmp/ipykernel_779/999185835.py:37: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[col] = pd.to_datetime(df[col], errors="coerce")
/tmp/ipykernel_779/999185835.py:37: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[col] = pd.to_datetime(df[col], errors="coerce")




Processing: /content/batch3_kaggle/raw/01_stock_market_data/stock_market_data/sp500/csv/AAP.csv


/tmp/ipykernel_779/999185835.py:37: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[col] = pd.to_datetime(df[col], errors="coerce")



Processing: /content/batch3_kaggle/raw/01_stock_market_data/stock_market_data/sp500/csv/AAPL.csv


...


Processing: /content/batch3_kaggle/raw/02_sp500_financials_news/02_company_news_sentiment.csv


/tmp/ipykernel_779/999185835.py:37: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors="coerce")



Processing: /content/batch3_kaggle/raw/02_sp500_financials_news/03_company_balance_sheet.csv

Processing: /content/batch3_kaggle/raw/02_sp500_financials_news/04_company_financials.csv

Processing: /content/batch3_kaggle/raw/02_sp500_financials_news/05_company_filings.csv

Processing: /content/batch3_kaggle/raw/02_sp500_financials_news/06_company_officers.csv

Processing: /content/batch3_kaggle/raw/02_sp500_financials_news/price_data/A.csv

Processing: /content/batch3_kaggle/raw/02_sp500_financials_news/price_data/AAL.csv

Processing: /content/batch3_kaggle/raw/02_sp500_financials_news/price_data/AAP.csv

Processing: /content/batch3_kaggle/raw/02_sp500_financials_news/price_data/AAPL.csv

Processing: /content/batch3_kaggle/raw/02_sp500_financials_news/price_data/ABBV.csv

Processing: /content/batch3_kaggle/raw/02_sp500_financials_news/price_data/ABNB.csv

Processing: /content/batch3_kaggle/raw/02_sp500_financials_news/price_data/ABT.csv

Processing: /content/batch3_kaggle/raw/02_sp500_

/tmp/ipykernel_779/999185835.py:68: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = pd.to_numeric(df[col], errors="coerce")



Batch 3-ready files created.


In [ ]:
# ============================================================
# CELL 7: Zip each dataset folder and a master folder, then download
# ============================================================

import shutil
from google.colab import files

# Step 1:
# Create one zip per dataset folder.
dataset_zip_paths = []

for short_name in DATASETS:
    folder_to_zip = READY / short_name
    zip_base = ZIPS / short_name
    zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=str(folder_to_zip))
    dataset_zip_paths.append(zip_path)
    print("Created:", zip_path)

# Step 2:
# Create one master zip containing all Batch 3-ready outputs.
master_zip = shutil.make_archive(str(ZIPS / "batch3_ready_all"), "zip", root_dir=str(READY))
print("Created master zip:", master_zip)

# Step 3:
# Download strategy:
# - Start by downloading the individual dataset zips.
# - If the master zip is manageable, download that too.
#
# Uncomment whichever downloads you want.

# for zp in dataset_zip_paths:
#     files.download(zp)

# files.download(master_zip)

print("\nDone. You can now download the zip(s) from the file browser or uncomment files.download(...).")


Created: /content/batch3_kaggle/zips/01_stock_market_data.zip
Created: /content/batch3_kaggle/zips/02_sp500_financials_news.zip
Created: /content/batch3_kaggle/zips/03_sp500_companies_financial_info.zip
Created: /content/batch3_kaggle/zips/04_sp500_stocks.zip
Created master zip: /content/batch3_kaggle/zips/batch3_ready_all.zip

Done. You can now download the zip(s) from the file browser or uncomment files.download(...).


In [ ]:
# ============================================================
# CELL: Combine all cleaned stock_market_data files into one CSV
# ============================================================

# Goal:
# Take all per-ticker cleaned CSV files from Dataset 1
# and combine them into one staging CSV that is easier
# to load into PostgreSQL later.
#
# This is recommended as a SOURCE-SPECIFIC staging file.
# It is NOT meant to replace your final normalized price_daily table.

from pathlib import Path
import pandas as pd

# Step 1:
# Point to the folder that contains the cleaned Dataset 1 files.
SRC_DIR = Path("/content/batch3_kaggle/batch3_ready/01_stock_market_data")

# Step 2:
# Choose an output file name for the combined staging CSV.
OUT_FILE = SRC_DIR / "stock_market_data_sp500_combined.csv"

# Step 3:
# Find all cleaned per-ticker CSV files.
# We only want files ending in "__cleaned.csv".
cleaned_files = sorted(SRC_DIR.glob("*__cleaned.csv"))

print(f"Found {len(cleaned_files)} cleaned files.")

frames = []

# Step 4:
# Read each file, infer ticker from the file name,
# standardize column names a bit, and append it to a list.
for fp in cleaned_files:
    df = pd.read_csv(fp)

    # Infer ticker from file name:
    # e.g. "aapl__cleaned.csv" -> "AAPL"
    ticker = fp.stem.replace("__cleaned", "").upper()

    # Add ticker + source columns so the combined file
    # still keeps useful provenance.
    df["ticker"] = ticker
    df["source_name"] = "stock_market_data_sp500"
    df["source_file"] = fp.name

    # Rename columns to match your target schema more closely.
    if "date" in df.columns:
        df = df.rename(columns={"date": "trading_date"})
    if "adjusted_close" in df.columns and "adj_close" not in df.columns:
        df = df.rename(columns={"adjusted_close": "adj_close"})

    # Reorder columns so the output is cleaner.
    preferred_order = [
        "ticker",
        "trading_date",
        "open",
        "high",
        "low",
        "close",
        "adj_close",
        "volume",
        "source_name",
        "source_file",
    ]

    ordered_cols = [c for c in preferred_order if c in df.columns]
    extra_cols = [c for c in df.columns if c not in ordered_cols]
    df = df[ordered_cols + extra_cols]

    frames.append(df)

# Step 5:
# Combine all per-ticker DataFrames into one big DataFrame.
combined = pd.concat(frames, ignore_index=True)

# Step 6:
# Final cleanup for the combined staging file.
if "trading_date" in combined.columns:
    combined["trading_date"] = pd.to_datetime(
        combined["trading_date"], errors="coerce"
    )

# Drop rows missing the fields you definitely need.
required_cols = [c for c in ["ticker", "trading_date", "close"] if c in combined.columns]
if required_cols:
    combined = combined.dropna(subset=required_cols)

# Optional de-dup:
# Since each file is one ticker, this mainly protects against
# accidental duplicate rows after concatenation.
dedupe_cols = [c for c in [
    "ticker", "trading_date", "open", "high", "low", "close", "adj_close", "volume"
] if c in combined.columns]

if dedupe_cols:
    combined = combined.drop_duplicates(subset=dedupe_cols)

# Sort for sanity.
sort_cols = [c for c in ["ticker", "trading_date"] if c in combined.columns]
if sort_cols:
    combined = combined.sort_values(sort_cols)

# Step 7:
# Save the combined CSV.
combined.to_csv(OUT_FILE, index=False)

print(f"Combined file saved to: {OUT_FILE}")
print(f"Final shape: {combined.shape}")
display(combined.head())

Found 409 cleaned files.
Combined file saved to: /content/batch3_kaggle/batch3_ready/01_stock_market_data/stock_market_data_sp500_combined.csv
Final shape: (2515911, 10)


,ticker,trading_date,open,high,low,close,adj_close,volume,source_name,source_file
0,A,1999-11-18,32.546494,35.765381,28.612303,31.473534,26.929760,62546380.0,stock_market_data_sp500,a__cleaned.csv
1,A,1999-11-19,30.713518,30.758226,28.478184,28.880545,24.711119,15234146.0,stock_market_data_sp500,a__cleaned.csv
2,A,1999-11-22,29.551144,31.473534,28.657009,31.473534,26.929760,6577870.0,stock_market_data_sp500,a__cleaned.csv
3,A,1999-11-23,30.400572,31.205294,28.612303,28.612303,24.481602,5975611.0,stock_market_data_sp500,a__cleaned.csv
4,A,1999-11-24,28.701717,29.998213,28.612303,29.372318,25.131901,4843231.0,stock_market_data_sp500,a__cleaned.csv


In [ ]:
# ============================================================
# COLAB CELL: Build a load-ready Dataset 1 CSV with only 2016+ rows
# ============================================================

import pandas as pd
from pathlib import Path
import os

# Step 1:
# Point to the combined Dataset 1 CSV you already made earlier.
SRC = Path("/content/batch3_kaggle/batch3_ready/01_stock_market_data/stock_market_data_sp500_combined.csv")

# Step 2:
# Choose the output path for the smaller, database-load-ready file.
OUT = SRC.with_name("d1_prices_load_2016plus.csv")

# Step 3:
# Define the cutoff date and chunk size.
# We use chunking so Colab does not have to hold the whole file in memory.
CUTOFF = pd.Timestamp("2016-01-01")
CHUNK_SIZE = 200_000

# Step 4:
# Read only the header first so we can detect column names safely.
header = pd.read_csv(SRC, nrows=0)

# Step 5:
# Figure out what the date column is called.
# Your combined file should already have trading_date, but this makes the cell robust.
date_col = "trading_date" if "trading_date" in header.columns else "date"

# Step 6:
# Figure out what the ticker column is called.
ticker_col = "ticker" if "ticker" in header.columns else "symbol"

# Step 7:
# Decide which columns we actually want in the DB load file.
# This matches the PostgreSQL staging table stg_d1_prices_raw.
keep_cols = [
    c for c in [
        ticker_col,
        date_col,
        "open",
        "high",
        "low",
        "close",
        "adj_close",
        "volume",
        "source_name",
        "source_file",
    ]
    if c in header.columns
]

print("Using date column :", date_col)
print("Using ticker column:", ticker_col)
print("Keeping columns    :", keep_cols)

# Step 8:
# Initialize counters and trackers for QA.
rows_in = 0
rows_out = 0
first_write = True
unique_tickers = set()
min_kept = None
max_kept = None

# Step 9:
# Process the file in chunks.
for chunk in pd.read_csv(SRC, chunksize=CHUNK_SIZE, low_memory=False):
    rows_in += len(chunk)

    # Parse the date column safely.
    chunk[date_col] = pd.to_datetime(chunk[date_col], errors="coerce")

    # Keep only 2016-01-01 and later.
    chunk = chunk[chunk[date_col] >= CUTOFF].copy()

    # Rename columns to the exact names the staging table expects.
    if ticker_col != "ticker":
        chunk = chunk.rename(columns={ticker_col: "ticker"})
    if date_col != "trading_date":
        chunk = chunk.rename(columns={date_col: "trading_date"})

    # Keep only the expected load columns, in the expected order.
    final_cols = [
        c for c in [
            "ticker",
            "trading_date",
            "open",
            "high",
            "low",
            "close",
            "adj_close",
            "volume",
            "source_name",
            "source_file",
        ]
        if c in chunk.columns
    ]
    chunk = chunk[final_cols]

    # Track some quick QA stats.
    if not chunk.empty:
        rows_out += len(chunk)
        if "ticker" in chunk.columns:
            unique_tickers.update(
                chunk["ticker"].dropna().astype(str).str.upper().tolist()
            )

        chunk_min = chunk["trading_date"].min()
        chunk_max = chunk["trading_date"].max()

        if min_kept is None or chunk_min < min_kept:
            min_kept = chunk_min
        if max_kept is None or chunk_max > max_kept:
            max_kept = chunk_max

        # Write the filtered chunk to disk.
        chunk.to_csv(
            OUT,
            mode="w" if first_write else "a",
            header=first_write,
            index=False
        )
        first_write = False

# Step 10:
# Print the summary so you can sanity-check the result.
print("\nDone.")
print(f"Rows before filter : {rows_in:,}")
print(f"Rows after filter  : {rows_out:,}")
print(f"Rows removed       : {rows_in - rows_out:,}")
print(f"Unique tickers kept: {len(unique_tickers):,}")
print(f"Date range kept    : {min_kept} to {max_kept}")
print(f"Output file        : {OUT}")
print(f"Output file size   : {os.path.getsize(OUT) / (1024 * 1024):.2f} MB")

# Step 11:
# Optional: preview the first few rows of the filtered file.
preview = pd.read_csv(OUT, nrows=5)
display(preview)

Using date column : trading_date
Using ticker column: ticker
Keeping columns    : ['ticker', 'trading_date', 'open', 'high', 'low', 'close', 'adj_close', 'volume', 'source_name', 'source_file']

Done.
Rows before filter : 2,515,911
Rows after filter  : 547,498
Rows removed       : 1,968,413
Unique tickers kept: 409
Date range kept    : 2016-01-02 00:00:00 to 2022-12-12 00:00:00
Output file        : /content/batch3_kaggle/batch3_ready/01_stock_market_data/d1_prices_load_2016plus.csv
Output file size   : 80.26 MB


,ticker,trading_date,open,high,low,close,adj_close,volume,source_name,source_file
0,A,2016-01-04,41.060001,41.189999,40.340000,40.689999,38.468792,3287300.0,stock_market_data_sp500,a__cleaned.csv
1,A,2016-01-05,40.730000,40.950001,40.340000,40.549999,38.336441,2587200.0,stock_market_data_sp500,a__cleaned.csv
2,A,2016-01-06,40.240002,40.990002,40.049999,40.730000,38.506611,2103600.0,stock_market_data_sp500,a__cleaned.csv
3,A,2016-01-07,40.139999,40.150002,38.810001,39.000000,36.871059,3504300.0,stock_market_data_sp500,a__cleaned.csv
4,A,2016-01-08,39.220001,39.709999,38.470001,38.590000,36.483444,3736700.0,stock_market_data_sp500,a__cleaned.csv


## What you should for Batch 3

From the `batch3_ready` outputs, send in this order:

### Batch 3A
- each `*_inventory.csv`
- each `*_cleaning_log.csv`
- the `__sample.csv` files

### Batch 3B
- the smaller `__cleaned.csv` files

### Batch 3C
- any `__chunk_partXX.csv` files for the large tables

This keeps the intake clean and makes entity-resolution review much easier.
